In [23]:
from src.model import SFR
from transformers import AutoTokenizer
import torch
from datasets import load_dataset
from src.eval.run_eval import prepare_retrieval_embeds

In [3]:
device = "cuda"

In [4]:
model_id = 'salesforce/sfr-embedding-mistral'
model = SFR.from_pretrained(model_id,torch_dtype = torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_id)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
model.eval()
model.to(device)

SFR(
  (embed_tokens): Embedding(32000, 4096, padding_idx=2)
  (layers): ModuleList(
    (0-31): 32 x MistralDecoderLayer(
      (self_attn): MistralAttention(
        (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
      )
      (mlp): MistralMLP(
        (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
        (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
        (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
    )
  )
  (norm): MistralRMSNorm((4096,), eps=1e-05)
  (rotary_emb): MistralRotaryEmbedding()
)

In [7]:
ds = load_dataset("brimmann2/squad_qa1", split="train")

In [18]:
documents = ds.select(range(3))["text"]

In [44]:
documents_list = [[s] for s in list(documents)]

In [53]:
num_samples = len(documents_list)
original_orders = []
for idx,background in enumerate(documents_list):
    original_orders.extend(
            [idx] * len(background)
        )

In [ ]:
documents_as_strings_list = [x for y in documents_list for x in y]

In [58]:
r = prepare_retrieval_embeds(list(documents_as_strings_list), model, tokenizer, batch_size=1)

In [61]:
retrieval_embeds = [[] for _ in range(num_samples)]

In [ ]:
assert len(r) == len(original_orders)

In [66]:
for id, embeds in zip(original_orders,r):
    print(id, embeds)

0 tensor([-1.3906, -3.3125, -5.5000,  ...,  8.0625,  0.8750,  5.2500],
       dtype=torch.bfloat16)
1 tensor([ 3.6719, -3.1406, -0.9180,  ...,  8.5625, -5.1562, -0.9766],
       dtype=torch.bfloat16)
2 tensor([ 3.4688,  1.3828, -0.2021,  ...,  3.2344, -4.9375,  4.1875],
       dtype=torch.bfloat16)


In [67]:
for id,embeds in zip(original_orders,r):
            retrieval_embeds[id].append(embeds)

In [1]:
from src.eval.symantic.embedding_utils import get_documents_embeds

In [3]:
r = get_documents_embeds('brimmann2/squad_qa1', "train")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [4]:
r

[[tensor([-1.3906, -3.3125, -5.5000,  ...,  8.0625,  0.8750,  5.2500],
         dtype=torch.bfloat16)],
 [tensor([ 3.6719, -3.1406, -0.9180,  ...,  8.5625, -5.1562, -0.9766],
         dtype=torch.bfloat16)],
 [tensor([ 3.4688,  1.3828, -0.2021,  ...,  3.2344, -4.9375,  4.1875],
         dtype=torch.bfloat16)]]